# 3. Statistiek van de hoogwaterstand, hoofdstations

Python-versie van het R-script **`statistics-wl10-impr.Rmd`** (KNMI, versie februari 2026).

Dit notebook schat herhalingstijdwaarden (return values) van de hoogwaterstand, omnidirectioneel en per
windrichtingssector, voor Vlissingen, Hoek van Holland, IJmuiden, Den Helder, Harlingen, Delfzijl en Hansweert:

1. inlezen van de gemeten hoogwaterstanden (gecorrigeerd voor de trend) en koppelen aan de gemeten windrichting;
2. fitten van Generalised Weibull-staarten (GW) aan de modeldata van SEAS5-DCSM5, en schatten van de onzekerheid
   van de vormparameter met een bootstrap;
3. fitten van GW-staarten aan de metingen, met de vormparameter uit SEAS5-DCSM5;
4. schatten van de onzekerheid met een bootstrap van de meetjaren, zonder en met modelfout;
5. gladstrijken van de richtingsafhankelijkheid. Het resultaat (`meas3`) wordt gebruikt voor de tabellen in
   `6_production_wl.ipynb`.

De resultaten worden als `.pkl`-bestanden in `output_path` geschreven, met dezelfde namen en
variabelen als de `.RData`-bestanden van het R-script.

Elk station krijgt precies de toevalsgetallen die het in de R-run krijgt
(`set.seed(342911)`, alle stations in de volgorde van R). Daarom geeft dit notebook dezelfde resultaten als het
R-script, inclusief de bootstrap-trekkingen, ook als je maar één station of de stations in een andere volgorde
doorrekent. Voorwaarde: `nb` en `r_stations` zijn gelijk aan die van de R-run.

Rekentijd: ongeveer 1 minuut per bootstrap-trekking per station voor de bootstrap van SEAS5-DCSM5.

Volgend notebook: `4_statistics_wl_others.ipynb`.

## Instellingen
Pas deze zo nodig aan. In `stations` mag je elke selectie van stations zetten, in elke volgorde.
`r_stations` is de stationslijst van de R-run: daaruit volgen de toevalsgetallen, dus die pas je normaal niet aan.
De standaardwaarden van de mappen, van `nb` en van `r_stations` staan in `kuststatistiek/settings.py`.

In [ ]:
from kuststatistiek import settings

input_path = settings.DATA_PATH  # map met de aangeleverde invoerdata
output_path = settings.OUTPUT_PATH  # map voor de resultaten
nb = settings.NB  # aantal bootstrap-trekkingen (R: nb <- 100)
r_stations = settings.WL_STATIONS  # stationslijst van de R-run, in de volgorde van R

# stations die je wilt doorrekenen (bijvoorbeeld alleen ["hoekvanh"])
stations = [
    "vlissing",
    "hoekvanh",
    "ijmuiden",
    "denhelde",
    "harlinge",
    "delfzijl",
    "hansweert",
]

## Voorbereiding
Laad de functies en de vaste instellingen van het R-script. `RRandom` is een kopie van de
toevalsgenerator van R, zodat de toevalsgetallen precies die van R zijn.

In [ ]:
from kuststatistiek.settings import (
    Paths,
    Bins,
    wl_settings,
    WL_STATIONS,
    WL_STATIONNAME,
    WL_WINDSTATIONS,
    WL_MEASFILE,
    PN,
)
from kuststatistiek.rstream import wl_random_streams
from kuststatistiek.io.read_wind import read_seas5_multi
from kuststatistiek.io.read_wl import read_station_wlmeas, link_wlmeas_windmeas
from kuststatistiek.analysis import (
    fit_seas5_tails,
    seas5_blocks,
    bootstrap_seas5,
    fit_meas_fixed_shape,
    meas_bootstrap_years,
    meas_bootstrap,
    model_error_draws,
    smooth_directionality,
)
from kuststatistiek.rutils import save_pkl, load_pkl, r_apply_sd
from kuststatistiek.plotting import plot_returnlevels

for station in stations:
    if station not in r_stations:
        raise ValueError(f"station {station} staat niet in r_stations")

paths = Paths(input_path=input_path, output_path=output_path)
WL = wl_settings(nb=nb)  # constanten van het R-script (chunk 'fixed')
bins = Bins()  # richtingssectoren
windstation = dict(
    zip(WL_STATIONS, WL_WINDSTATIONS)
)  # KNMI-windstation dat bij elk getijstation hoort

save_pkl(
    paths.out("statistics-wl10settings.pkl"),
    stationsys5=r_stations,
    stationname=WL_STATIONNAME,
    freq=WL.freq,
    EIvalue=WL.EIvalue,
    delt=WL.delt,
    factor=WL.factor,
    pn=PN,
    pthreshold=WL.pthreshold,
    kthresh=WL.kthresh,
    mthresh=WL.mthresh,
    x=WL.x,
)

## Toevalsgetallen van R per station
In R komen alle toevalsgetallen uit één reeks, die stap na stap en binnen een stap station na station wordt
gebruikt. Hier wordt die reeks voor alle stations van `r_stations` nagespeeld (alleen het trekken, zonder fits;
dit duurt ongeveer een halve minuut). Voor elk station en elke stap wordt bewaard waar de reeks van dat station
begint. Zo krijgt elk station precies de toevalsgetallen van de R-run, welke stations je ook doorrekent.

In [ ]:
streams = wl_random_streams(
    WL.seed, r_stations, nb, paths.input_path
)  # begint zelf met set.seed(342911)

## Stap 1 – Gemeten hoogwaterstanden (R-chunk `eimeas`)
Lees de gemeten hoogwaterstanden in, gecorrigeerd voor de (gladde) trend. Net als in R wordt een heel klein
toevalsgetal opgeteld, zodat er geen gelijke waarden zijn.

Uitvoer: `<station>_multi_meas.pkl`

In [ ]:
wl_measdata = {}
for station in stations:
    a = read_station_wlmeas(station, paths.input_path, WL_MEASFILE)
    wlm = (
        a["wspeed"]
        + (streams[station]["eimeas"].runif(len(a["wspeed"])) - 0.5) * 0.0001
    )
    wl_measdata[station] = dict(
        wlm=wlm, dtm=a["t"].to_numpy(), Nm=len(wlm), hwtrend=a["hwtrend"]
    )
    save_pkl(paths.out(f"{station}_multi_meas.pkl"), **wl_measdata[station])
    print(f"{station}: {len(wlm)} hoogwaters")

## Stap 2 – Hoogwaterstanden koppelen aan de gemeten windrichting (R-chunk `dir-estimate-meas2`)
Bij elk hoogwater wordt de windrichting genomen uit de uurmetingen van een nabijgelegen KNMI-station
(vectorgemiddelde van de laatste 3 uur).

Uitvoer: `processed-wl-winddir-<station>.pkl`

In [ ]:
wlwinddir_measdata = {}
for station in stations:
    wlwinddir_measdata[station] = link_wlmeas_windmeas(
        wl_measdata[station]["wlm"],
        wl_measdata[station]["dtm"],
        windstation[station],
        paths.input_path,
    )
    save_pkl(
        paths.out(f"processed-wl-winddir-{station}.pkl"), **wlwinddir_measdata[station]
    )

## Stap 3 – GW-staarten uit SEAS5-DCSM5 (R-chunk `dir-es-seas5dcms5`)
Fit de Generalised Weibull-staart aan de hoogwaterstanden van SEAS5-DCSM5, omnidirectioneel en per
windrichtingssector.

Uitvoer: `<station>_stats_multi.pkl`

In [ ]:
statdata = {}
for station in stations:
    wl_dcsm = read_seas5_multi(station, paths.input_path)
    statdata[station] = fit_seas5_tails(
        wl_dcsm["wl"], wl_dcsm["dwind"], WL, bins, caseId=station
    )
    save_pkl(paths.out(f"{station}_stats_multi.pkl"), **statdata[station])
    print(f"{station}: GW-fit SEAS5-DCSM5 klaar")

## Stap 4 – Bootstrap van SEAS5-DCSM5 (R-chunk `dir-estimate-bootstrap`)
Trek `nb` bootstrap-steekproeven uit de SEAS5-DCSM5-data (er worden hele blokken herbemonsterd) en herhaal
de GW-fit voor elke steekproef. Alleen de vormparameters worden bewaard: die worden gebruikt in stap 6 en 7.

Uitvoer: `<station>_stats_wl_bootdata.pkl`

In [ ]:
bootdata = {}
for station in stations:
    wl_dcsm = read_seas5_multi(station, paths.input_path)
    blockno, blocks = seas5_blocks(wl_dcsm["dt"], streams[station]["bootstrap"], nb)
    bs = bootstrap_seas5(
        wl_dcsm["wl"],
        wl_dcsm["dwind"],
        blockno,
        blocks,
        WL,
        bins,
        caseId=station,
        dir_threshold=WL.pthreshold[WL.kthresh - 1],
        keep_quantiles=False,
    )
    bootdata[station] = dict(
        theta0bs=bs["theta0bs"],
        theta0dirbs=bs["theta0dirbs"],
        theta0sd=r_apply_sd(bs["theta0bs"], 1),
        theta0dirsd=r_apply_sd(bs["theta0dirbs"], 1),
        blocks=blocks,
        blockno=blockno,
    )
    save_pkl(paths.out(f"{station}_stats_wl_bootdata.pkl"), **bootdata[station])
    print(f"{station}: bootstrap SEAS5-DCSM5 klaar")

## Stap 5 – GW-staarten uit de metingen (R-chunk `omni-estimates-meas`)
Fit de GW-staart aan de metingen, met de vormparameter vastgezet op de schatting uit SEAS5-DCSM5 (stap 3).

Uitvoer: `<station>_stats_multi_meas0.pkl` en `stats_omni_meas.pkl` (een lijst met de fits van alle stations
van `r_stations`, zoals in R; bij een selectie van stations worden alleen die stations in de lijst bijgewerkt)

In [ ]:
f_omni = paths.out("stats_omni_meas.pkl")
esmeasGW = load_pkl(f_omni)["esmeasGW"] if f_omni.exists() else []
if len(esmeasGW) != len(r_stations):
    esmeasGW = [None] * len(r_stations)

meas0 = {}
for station in stations:
    meas0[station], es = fit_meas_fixed_shape(
        statdata[station],
        wl_measdata[station]["wlm"],
        wlwinddir_measdata[station]["wlmm"],
        wlwinddir_measdata[station]["winddir"],
        WL,
        bins,
        caseId=station,
    )
    esmeasGW[r_stations.index(station)] = es
    save_pkl(paths.out(f"{station}_stats_multi_meas0.pkl"), **meas0[station])
save_pkl(f_omni, esmeasGW=esmeasGW)

## Stap 6 – Bootstrap van de metingen (R-chunk `dir-estimate-meas-bootstrap`)
Trek `nb` bootstrap-steekproeven van de meetjaren en herhaal de fit van stap 5 voor elke steekproef, met de
vormparameters uit de bootstrap van SEAS5-DCSM5 (stap 4), zonder modelfout.
(Het resultaat `meas1` wordt in R alleen voor figuren gebruikt.)

Uitvoer: `<station>_meas_wl_bootdata.pkl` (de steekproeven) en `<station>_stats_multi_meas1.pkl`

In [ ]:
# variabelen van meas0 die worden overgenomen in meas1 en meas2
meas0_keys = [
    "esmeasGWpure",
    "esmeasGWfull",
    "x",
    "mum",
    "Mum",
    "pbin0m",
    "mum0",
    "Mum0",
    "qmum",
]

measbootdata = {}
for station in stations:
    m, mm = wl_measdata[station], wlwinddir_measdata[station]
    measbootdata[station] = meas_bootstrap_years(
        m["wlm"],
        m["dtm"],
        mm["wlmm"],
        mm["dtmm"],
        mm["winddir"],
        streams[station]["meas_bootstrap"],
        nb,
    )
    save_pkl(paths.out(f"{station}_meas_wl_bootdata.pkl"), **measbootdata[station])

    r = meas_bootstrap(
        measbootdata[station],
        bootdata[station]["theta0bs"],
        bootdata[station]["theta0dirbs"],
        WL,
        bins,
        caseId=station,
        pure=True,
    )
    m0 = meas0[station]
    save_pkl(
        paths.out(f"{station}_stats_multi_meas1.pkl"),
        **{k: m0[k] for k in meas0_keys},
        qmbs=r["qmbs"],
        qmpurebs=r["qmpurebs"],
        thetampurebs=r["thetampurebs"],
        freq=WL.freq,
        Qfreqm=m0["Qfreqm"],
        Qfreq0m=m0["Qfreq0m"],
        Qmbs=r["Qmbs"],
        Q0mbs=r["Q0mbs"],
        Qmsd=r["Qmsd"],
        Q0msd=r["Q0msd"],
    )

## Stap 7 – Bootstrap met modelfout (R-chunk `dir-estimate-meas-bootstrap-moderr`)
Hetzelfde als stap 6, maar eerst wordt een toevallige modelfout (standaardafwijking `modelstd` = 0,1)
opgeteld bij de vormparameters.

Uitvoer: `<station>_stats_multi_meas2.pkl`

In [ ]:
meas2 = {}
for station in stations:
    theta00bs, theta00dirbs = model_error_draws(
        bootdata[station]["theta0bs"],
        bootdata[station]["theta0dirbs"],
        WL,
        streams[station]["model_error"],
    )
    r = meas_bootstrap(
        measbootdata[station], theta00bs, theta00dirbs, WL, bins, caseId=station
    )
    m0 = meas0[station]
    meas2[station] = dict(
        **{k: m0[k] for k in meas0_keys},
        theta00bs=theta00bs,
        theta00dirbs=theta00dirbs,
        qmbs=r["qmbs"],
        freq=WL.freq,
        Qfreqm=m0["Qfreqm"],
        Qfreq0m=m0["Qfreq0m"],
        Qmbs=r["Qmbs"],
        Q0mbs=r["Q0mbs"],
        Qmsd=r["Qmsd"],
        Q0msd=r["Q0msd"],
    )
    save_pkl(paths.out(f"{station}_stats_multi_meas2.pkl"), **meas2[station])
    print(f"{station}: bootstrap met modelfout klaar")

## Stap 8 – Richtingsafhankelijkheid gladstrijken (R-chunk `combine`)
Combineer de omnidirectionele schattingen uit de metingen met de richtingsafhankelijkheid uit SEAS5-DCSM5,
gladgestreken over de richtingssectoren. Dit is het eindresultaat dat voor de tabellen wordt gebruikt.

Uitvoer: `<station>_stats_multi_meas3.pkl`

In [ ]:
meas3 = {}
for station in stations:
    meas3[station] = smooth_directionality(statdata[station], meas2[station], WL, bins)
    save_pkl(paths.out(f"{station}_stats_multi_meas3.pkl"), **meas3[station])
    print(f"{station} klaar")

## Figuur (controle)
De uiteindelijke herhalingstijdwaarden van de hoogwaterstand per richting, voor enkele herhalingstijden, per station.
(De overige figuren van het R-script zijn hier niet overgenomen.)

In [ ]:
for station in stations:
    plot_returnlevels(
        meas3[station]["Qfreq0m"],
        bins.cats0,
        WL.freq,
        title=station,
        ylabel="hoogwaterstand [m]",
    )